In [10]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "eval" / "retrieval_smoke.py").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not find the mini-rag project root")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from eval.retrieval_smoke import load_questions_async
from openai import AsyncOpenAI
from rich_struct import print_json

from mini_rag.ask import ask_async, print_answer
from mini_rag.config import DEFAULT_EMBEDDING_MODEL
from mini_rag.llm import OpenAIAnswerGenerator, OpenAIRetriever
from mini_rag.models import Answer

In [11]:
from mini_rag.config import DEFAULT_LLM_MODEL


async def answer_question(question: str) -> Answer:
    async with AsyncOpenAI() as client:
        retriever = OpenAIRetriever(Path("../index") / DEFAULT_EMBEDDING_MODEL, client)
        await retriever.load_index(Path("../index/chunks_600_90.jsonl"))

        generator = OpenAIAnswerGenerator(client, DEFAULT_LLM_MODEL)

        answer = await ask_async(question, retriever, generator, 5)
        return answer

In [12]:
questions = [q async for q in load_questions_async(Path("../data/questions.jsonl"))]

In [13]:
question = questions[1].question
print(f"Question: {question}")

answer_ok = await answer_question(question)

Question: What is the difference between climate mitigation and adaptation?


09:28:36.338 INFO     Loading index data from ..\index\text-embedding-3-small
09:28:36.343 INFO     Loading embeddings manifest from ..\index\text-embedding-3-small\embedding_manifest.json
09:28:36.378 INFO     Loading normalized embeddings from ..\index\text-embedding-3-small\embeddings.npy
09:28:36.410 INFO     Reading chunks from ..\index\chunks_600_90.jsonl
09:28:38.387 INFO     Retrieving closest semanticly chunks for user question
09:28:38.391 DEBUG    Searching for text What is the difference between climate mitigation and adaptation?
09:28:38.394 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
09:28:38.719 DEBUG    OpenAI returned 1 embeddings (total_tokens: 10) [Batch: 1]
09:28:38.723 DEBUG    Semantic searching
09:28:38.734 INFO     Building question context
09:28:38.738 INFO     Getting answer for user question from LLM
09:28:38.741 INFO     Quering OpenAI's model 'gpt-5.4-mini'
09:28:40.508 INFO     OpenAI model 'gpt-5.4-mini' responded in 1.76 seconds (input_tokens: 2726, output_tokens: 80, total_tokens: 2806) [Response: resp_0537badaeb1d7da7006aff9436702c87d0ba113a870503d182]

In [14]:
print_answer(answer_ok)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Climate mitigation refers to actions that reduce greenhouse gas emissions or otherwise address the causes of    │
│ climate change, while adaptation refers to actions that adjust human or natural systems to actual or expected   │
│ climate effects in order to lessen harm or take advantage of benefits. [1][5]                                   │
│                                                                                                                 │
│  ref  document_id  page  page_end                                                                               │
│  1    WGIII_CH14   47    47                                                                                     │
│  5    WGIII_CH04   60    60                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [15]:
print_json(answer_ok.model_dump_json())

{                                                                                                                  
  "answer": "Climate mitigation refers to actions that reduce greenhouse gas emissions or otherwise address the cau
  "citations": [                                                                                                   
    {                                                                                                              
      "ref": 1,                                                                                                    
      "chunk_id": "WGIII_CH14_p047_c002",                                                                          
      "document_id": "WGIII_CH14",                                                                                 
      "page": 47,                                                                                                  
      "page_end": 47                                                                                               
    },                                                                                                             
    {                                                                                                              
      "ref": 5,                                                                                                    
      "chunk_id": "WGIII_CH04_p060_c003",                                                                          
      "document_id": "WGIII_CH04",                                                                                 
      "page": 60,                                                                                                  
      "page_end": 60                                                                                               
    }                                                                                                              
  ],                                                                                                               
  "insufficient_evidence": false,                                                                                  
  "retrieved_chunk_ids": [                                                                                         
    "WGIII_CH14_p047_c002",                                                                                        
    "WGIII_CH04_p061_c002",                                                                                        
    "WGIII_CH17_p032_c003",                                                                                        
    "WGIII_CH13_p049_c001",                                                                                        
    "WGIII_CH04_p060_c003"                                                                                         
  ]                                                                                                                
}                                                                                                                  

In [16]:
question = questions[28].question
print(f"Question: {question}")

answer_fail = await answer_question(question)

Question: How much did Apple earn in fiscal year 2025?


09:28:40.596 INFO     Loading index data from ..\index\text-embedding-3-small
09:28:40.601 INFO     Loading embeddings manifest from ..\index\text-embedding-3-small\embedding_manifest.json
09:28:40.634 INFO     Loading normalized embeddings from ..\index\text-embedding-3-small\embeddings.npy
09:28:40.663 INFO     Reading chunks from ..\index\chunks_600_90.jsonl
09:28:42.503 INFO     Retrieving closest semanticly chunks for user question
09:28:42.507 DEBUG    Searching for text How much did Apple earn in fiscal year 2025?
09:28:42.511 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
09:28:43.610 DEBUG    OpenAI returned 1 embeddings (total_tokens: 12) [Batch: 1]
09:28:43.617 DEBUG    Semantic searching
09:28:43.629 INFO     Building question context
09:28:43.635 INFO     Getting answer for user question from LLM
09:28:43.641 INFO     Quering OpenAI's model 'gpt-5.4-mini'
09:28:45.531 INFO     OpenAI model 'gpt-5.4-mini' responded in 1.89 seconds (input_tokens: 3145, output_tokens: 38, total_tokens: 3183) [Response: resp_03578879b705edb4006aff943b5aec87d0b25a0de64364aa06]

In [17]:
print_answer(answer_fail)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ I'm sorry but based on provided document fragments I can't answer the question                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [18]:
print_json(answer_fail.model_dump_json())

{                                                                                                                  
  "answer": "I'm sorry but based on provided document fragments I can't answer the question",                      
  "citations": [],                                                                                                 
  "insufficient_evidence": true,                                                                                   
  "retrieved_chunk_ids": [                                                                                         
    "WGIII_CH11_p058_c002",                                                                                        
    "WGIII_CH15_p011_c002",                                                                                        
    "WGIII_CH15_p026_c002",                                                                                        
    "WGIII_CH16_p063_c003",                                                                                        
    "WGIII_CH15_p046_c001"                                                                                         
  ]                                                                                                                
}                                                                                                                  